In [ ]:
# In this file, we convert QTSA to SFT format converter
# This script converts QTSA data (question, thinking, solution, answer) into the recommended SFT format: {"instruction": ..., "response": ...}

In [ ]:
import json
from transformers import AutoTokenizer

def detailed_token_analysis(jsonl_file_paths):
    tokenizer = AutoTokenizer.from_pretrained("Qwen-0.6B-local")
    
    for file_path in jsonl_file_paths:
        print(f"\n{'='*50}")
        print(f"Analyzing: {file_path}")
        print(f"{'='*50}")
        
        total_combined_tokens = 0
        count = 0
        
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                for line_num, line in enumerate(file, 1):
                    try:
                        data = json.loads(line.strip())
                        count += 1

                        combined_text = f"{data.get('question', '')} {data.get('thinking', '')} {data.get('solution', '')} {data.get('answer', '')}"
                        combined_tokens = tokenizer.encode(combined_text)
                        total_combined_tokens += len(combined_tokens)
                        
                    except json.JSONDecodeError:
                        print(f"⚠️ JSON decode error in {file_path} at line {line_num}: {line[:100]}...")
                        continue
                    except Exception as e:
                        print(f"⚠️ Error processing line {line_num} in {file_path}: {str(e)}")
                        continue
            
            if count > 0:
                print(f"Total samples: {count:,}")
                print(f"Total tokens: {total_combined_tokens:,}")
            else:
                print("No valid samples found in this file.")
                
        except FileNotFoundError:
            print(f"❌ File not found: {file_path}")
        except Exception as e:
            print(f"❌ Error reading file {file_path}: {str(e)}")

if __name__ == "__main__":
    jsonl_file_paths = [
        "validation_QTSA/mcQTSA_train.jsonl",
        "validation_QTSA/mcQTSA_val.jsonl", 
        "testbench/mcQTSA_test.jsonl"
    ]
    
    detailed_token_analysis(jsonl_file_paths)

In [ ]:
# QTSA → SFT format converter
# This script converts QTSA data (question, thinking, solution, answer)
# into the recommended SFT format: {"instruction": ..., "response": ...}
# keeping <think>...</think> and <answer>...</answer> tags intact.

import json
import os

def convert_qtsa_to_sft(input_path, output_path):
    """
    Convert QTSA-format data into SFT-style JSONL.
    Keeps <think> and <answer> tags.
    """
    converted = []
    num_missing_fields = {"thinking": 0, "solution": 0, "answer": 0}

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            sample = json.loads(line)

            q = sample.get("question", "").strip()
            t = sample.get("thinking", "").strip()
            s = sample.get("solution", "").strip()
            a = sample.get("answer", "").strip()

            # Count missing fields for logging
            if not t:
                num_missing_fields["thinking"] += 1
            if not s:
                num_missing_fields["solution"] += 1
            if not a:
                num_missing_fields["answer"] += 1

            # Build instruction-response pair
            instruction = f"Question: {q}"

            response_parts = []
            if t:
                response_parts.append(t)  # already includes <think> tags
            if s:
                response_parts.append(f"Solution: {s}")
            if a:
                response_parts.append(a)  # already includes <answer> tags

            response = "\n".join(response_parts)

            converted.append({
                "instruction": instruction,
                "response": response
            })

    # Save as JSONL
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for item in converted:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Conversion complete: {input_path} → {output_path}")
    print(f"Total samples: {len(converted)}")
    print(f"Missing fields summary: {num_missing_fields}")


# ======== Main Execution ========
if __name__ == "__main__":
    input_files = {
        "validation_QTSA/mcQTSA_train.jsonl": "validation_QTSA/mcQTSA_train_sft.jsonl",
        "validation_QTSA/mcQTSA_val.jsonl": "validation_QTSA/mcQTSA_val_sft.jsonl"
    }

    for in_file, out_file in input_files.items():
        convert_qtsa_to_sft(in_file, out_file)


In [ ]:
# QTSA → SFT format converter
# This script converts QTSA data (question, thinking, solution, answer)
# into the recommended SFT format: {"instruction": ..., "response": ...}
# keeping <think>...</think> and <answer>...</answer> tags intact.

import json
import os

def convert_qtsa_to_sft(input_path, output_path):
    """
    Convert QTSA-format data into SFT-style JSONL.
    Keeps <think> and <answer> tags.
    """
    converted = []
    num_missing_fields = {"thinking": 0, "solution": 0, "answer": 0}

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            sample = json.loads(line)

            q = sample.get("question", "").strip()
            t = sample.get("thinking", "").strip()
            s = sample.get("solution", "").strip()
            a = sample.get("answer", "").strip()

            # Count missing fields for logging
            if not t:
                num_missing_fields["thinking"] += 1
            if not s:
                num_missing_fields["solution"] += 1
            if not a:
                num_missing_fields["answer"] += 1

            # Build instruction-response pair
            instruction = f"Question: {q}"

            response_parts = []
            if t:
                response_parts.append(t)  # already includes <think> tags
            if s:
                response_parts.append(f"Solution: {s}")
            if a:
                response_parts.append(a)  # already includes <answer> tags

            response = "\n".join(response_parts)

            converted.append({
                "instruction": instruction,
                "response": response
            })

    # Save as JSONL
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for item in converted:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Conversion complete: {input_path} → {output_path}")
    print(f"Total samples: {len(converted)}")
    print(f"Missing fields summary: {num_missing_fields}")


# ======== Main Execution ========
if __name__ == "__main__":
    input_files = {
        "validation_QTSA/mcQTSA_train_fixed.jsonl": "validation_QTSA/mcQTSA_train_fixed_sft.jsonl",
        "validation_QTSA/mcQTSA_val_fixed.jsonl": "validation_QTSA/mcQTSA_val_fixed_sft.jsonl"
    }

    for in_file, out_file in input_files.items():
        convert_qtsa_to_sft(in_file, out_file)

In [ ]:
# QTSA → SFT format converter
# This script converts QTSA data (question, thinking, solution, answer)
# into the recommended SFT format: {"instruction": ..., "response": ...}
# keeping <think>...</think> and <answer>...</answer> tags intact.

import json
import os

def convert_qtsa_to_sft(input_path, output_path):
    """
    Convert QTSA-format data into SFT-style JSONL.
    Keeps <think> and <answer> tags.
    """
    converted = []
    num_missing_fields = {"thinking": 0, "solution": 0, "answer": 0}

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            sample = json.loads(line)

            q = sample.get("question", "").strip()
            t = sample.get("thinking", "").strip()
            s = sample.get("solution", "").strip()
            a = sample.get("answer", "").strip()

            # Count missing fields for logging
            if not t:
                num_missing_fields["thinking"] += 1
            if not s:
                num_missing_fields["solution"] += 1
            if not a:
                num_missing_fields["answer"] += 1

            # Build instruction-response pair
            instruction = f"Question: {q}"

            response_parts = []
            if t:
                response_parts.append(t)  # already includes <think> tags
            if s:
                response_parts.append(f"Solution: {s}")
            if a:
                response_parts.append(a)  # already includes <answer> tags

            response = "\n".join(response_parts)

            converted.append({
                "instruction": instruction,
                "response": response
            })

    # Save as JSONL
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for item in converted:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Conversion complete: {input_path} → {output_path}")
    print(f"Total samples: {len(converted)}")
    print(f"Missing fields summary: {num_missing_fields}")


# ======== Main Execution ========
if __name__ == "__main__":
    input_files = {
        "validation_QTSA_mix/QTSA_train.jsonl": "validation_QTSA_mix/QTSA_train_sft.jsonl",
        "validation_QTSA_mix/QTSA_val.jsonl": "validation_QTSA_mix/QTSA_val_sft.jsonl"
    }

    for in_file, out_file in input_files.items():
        convert_qtsa_to_sft(in_file, out_file)

In [ ]:
# QTSA → Three-part format converter
# This script converts QTSA data into format with three parts:
# {"instruction": ..., "thinking": ..., "response": ...}

import json
import os

def convert_qtsa_to_three_part(input_path, output_path):
    """
    Convert QTSA-format data into three-part JSONL:
    - instruction: question only
    - thinking: thinking content (with <think> tags)
    - response: solution + answer
    """
    converted = []
    num_missing_fields = {"thinking": 0, "solution": 0, "answer": 0}

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            sample = json.loads(line)

            q = sample.get("question", "").strip()
            t = sample.get("thinking", "").strip()
            s = sample.get("solution", "").strip()
            a = sample.get("answer", "").strip()

            # Count missing fields for logging
            if not t:
                num_missing_fields["thinking"] += 1
            if not s:
                num_missing_fields["solution"] += 1
            if not a:
                num_missing_fields["answer"] += 1

            # Build three-part format
            instruction = f"Question: {q}"
            thinking = t  
            
            
            response_parts = []
            if s:
                response_parts.append(f"Solution: {s}")
            if a:
                response_parts.append(a)
            response = "\n".join(response_parts)

            converted.append({
                "instruction": instruction,
                "thinking": thinking,
                "response": response
            })

    # Save as JSONL
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for item in converted:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Conversion complete: {input_path} → {output_path}")
    print(f"Total samples: {len(converted)}")
    print(f"Missing fields summary: {num_missing_fields}")


# ======== Main Execution ========
if __name__ == "__main__":
    input_files = {
        "validation_QTSA_mix/QTSA_train.jsonl": "validation_QTSA_mix/QTSA_train_thinking.jsonl",
        "validation_QTSA_mix/QTSA_val.jsonl": "validation_QTSA_mix/QTSA_val_thinking.jsonl"
    }

    for in_file, out_file in input_files.items():
        convert_qtsa_to_three_part(in_file, out_file)